# 🚀 LifeMate Agent — Colab / Kaggle Runner & Submission Generator
### Zalo AI Challenge 2026 — Voice-driven AI Agent

Notebook này cho phép bạn chạy toàn bộ luồng xử lý trên **Google Colab (GPU T4 miễn phí)** hoặc **Kaggle (2x T4)**:
1. **Part B (ASR)**: `faster-whisper large-v3-turbo` chuyển đổi giọng nói tiếng Việt thành văn bản.
2. **Part C (Agent LLM)**: `Qwen/Qwen3-4B-Instruct-2507` (4.02B params) suy luận và gọi công cụ (Tool Calling).
3. **Part A (Lead/Eval)**: Quản lý ToolEnv, trích xuất `tool_calls` & `state_hash`, kiểm tra 7 tầng bằng `SubmissionLinter` và xuất file nộp bài chuẩn.

## 1. Kiểm tra GPU & Kết nối Google Drive

In [ ]:
# 1. Kiểm tra GPU T4 (15-16GB VRAM)
!nvidia-smi

# 2. Kết nối Google Drive để lưu bài nộp và checkpoint
from google.colab import drive
drive.mount('/content/drive')

## 2. Cài đặt các thư viện phụ thuộc

In [ ]:
!pip install -q transformers>=4.45 accelerate torch faster-whisper sentencepiece huggingface_hub jiwer bitsandbytes


## 3. Clone / Cập nhật mã nguồn & Chuẩn bị dữ liệu

In [ ]:
import os
from pathlib import Path

# 1. Clone repository nếu chưa có trên Colab SSD
WORKSPACE = Path("/content/LifeMateAgent")
if not WORKSPACE.exists():
    !git clone https://github.com/Platypus27-coder/zalo-ai-challenge.git /content/LifeMateAgent

%cd /content/LifeMateAgent
!git checkout main
!git pull origin main

# 2. Giải nén dữ liệu public_test hoặc train+public_test từ thư mục LifeMateAgent trên Google Drive
DATA_DIR = WORKSPACE / "train+public_test"
POSSIBLE_ZIP_PATHS = [
    Path("/content/drive/MyDrive/LifeMateAgent/data/public_test.zip"),
    Path("/content/drive/MyDrive/LifeMateAgent/data/train+public_test.zip"),
    Path("/content/drive/MyDrive/LifeMateAgent/public_test.zip"),
    Path("/content/drive/MyDrive/public_test.zip"),
    Path("/content/drive/MyDrive/train+public_test.zip"),
]

if not (DATA_DIR / "public_test" / "public_test.jsonl").exists():
    zip_found = False
    for zip_path in POSSIBLE_ZIP_PATHS:
        if zip_path.exists():
            print(f"📦 Tìm thấy dữ liệu tại: {zip_path}")
            print("Đang giải nén vào /content/LifeMateAgent/...")
            !unzip -q -o "{zip_path}" -d /content/LifeMateAgent/
            zip_found = True
            break
    if not zip_found:
        print("⚠️ Không tìm thấy file zip dữ liệu tại Google Drive (LifeMateAgent/data/public_test.zip).")

if (DATA_DIR / "public_test" / "public_test.jsonl").exists():
    print("\n✅ Dữ liệu public_test (634 mẫu) đã sẵn sàng!")
else:
    print("\n❌ Chưa có thư mục train+public_test/public_test/")

## 4. Kiểm tra hợp chuẩn toàn bộ hệ thống (Self-Verification)

In [ ]:
!python scripts/verify_all.py

## 5. Chạy thử nghiệm khói (Smoke Test: 5 mẫu Public Test)
Chạy thử 5 mẫu đầu tiên để xác nhận Agent LLM (Qwen3-4B-Instruct-2507) nạp vào GPU T4 mượt mà, phân tích câu thoại và gọi tool chính xác.
- Chế độ --asr-mode precomputed: Dùng ngay kết quả phiên âm Part B chất lượng cao có sẵn (cực nhanh ~10s, tiết kiệm VRAM).
- Bạn có thể đổi sang --asr-mode whisper nếu muốn chạy trực tiếp từ file âm thanh.

In [ ]:
!python scripts/generate_submission.py \
    --split public_test \
    --max-samples 5 \
    --asr-mode precomputed \
    --agent-mode transformers \
    --model-name Qwen/Qwen3-4B-Instruct-2507 \
    --verbose

## 6. Chạy chính thức toàn bộ 634 mẫu Public Test & Tạo File Nộp Bài
Sau khi smoke test thành công, chạy toàn bộ tập public_test (634 mẫu).
- Sử dụng --asr-mode precomputed và Qwen3-4B-Instruct-2507 với cơ chế quản lý VRAM thông minh (SDPA Attention + auto cache purge).
- Nếu muốn siêu nhẹ (chỉ tốn ~3GB VRAM), bạn có thể thêm cờ --load-in-4bit.

In [ ]:
# Tạo thư mục chứa bài nộp
!mkdir -p /content/outputs

!python scripts/generate_submission.py \
    --split public_test \
    --asr-mode precomputed \
    --agent-mode transformers \
    --model-name Qwen/Qwen3-4B-Instruct-2507 \
    --output /content/outputs/submission_public_test.jsonl \
    --verbose

## 7. Sao lưu vĩnh viễn lên Google Drive & Tải về máy tính
Lưu bài nộp vào Google Drive để không bị mất khi đóng tab, và tự động tải file nộp bài về máy để upload lên Leaderboard của Zalo AI Challenge.

In [ ]:
import shutil
from google.colab import files
from pathlib import Path

sub_file = Path("/content/outputs/submission_public_test.jsonl")
if sub_file.exists():
    # 1. Lưu vào thư mục submissions của dự án trên Google Drive
    drive_dest = Path("/content/drive/MyDrive/LifeMateAgent/submissions")
    drive_dest.mkdir(parents=True, exist_ok=True)
    shutil.copy(sub_file, drive_dest / sub_file.name)
    print(f"✅ Đã sao lưu vĩnh viễn vào Google Drive: {drive_dest / sub_file.name}")
    
    # 2. Tải về máy tính qua trình duyệt
    print("⬇️ Đang tải file về máy tính của bạn...")
    files.download(str(sub_file))
else:
    print("❌ Không tìm thấy file bài nộp:", sub_file)